In [ ]:
import pandas as pd
import os
import plotly.express as px
from dash import Input, Output, dcc, html
from IPython.display import VimeoVideo
from dash import Dash
from scipy.stats.mstats import trimmed_var
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


In [ ]:
VimeoVideo("715724401", h="062cb7d8cb", width=600)


In [ ]:
VimeoVideo("715724313", h="711e785135", width=600)


In [ ]:
def wrangle(filepath):  
    """Read SCF data file into ``DataFrame``.
    
    Returns only credit fearful households whose net worth is less than $2 million.

    Parameters
    ----------
    filepath : str
        Location of CSV file.
    """
    # Load data
    df = pd.read_csv(filepath)
    # Create mask
    mask = (df['TURNFEAR']==1) & (df['NETWORTH'] < 2e6)
    df = df[mask]
    
    return df


In [ ]:
df = wrangle("data/SCFP2019.csv.gz")

print("df type:", type(df))
print("df shape:", df.shape)
df.head()


In [ ]:
VimeoVideo("715724244", h="41e32f352f", width=600)


In [ ]:
app = Dash(__name__)

print("app type:", type(app))


In [ ]:
VimeoVideo("715724173", h="21f2757631", width=600)


In [ ]:
app.layout = html.Div(
    [
    # Application title
    html.H1("Survey of Consumer Finances"),
    # Bar chart element
    html.H2("High Variance Features"),
    # Bar chart
    dcc.Graph(id='bar-chart'),
    dcc.RadioItems(
    options=[
        {"label": "trimmed", "value": True},
        {"label": "not trimmed", "value": False}],
    value = True,
    id="trim-button"
    ),
    html.H2("K-means Clustering"),
    html.H3("Number of Clusters (k)"),
    dcc.Slider(min=2, max=12, step=1, value=2, id="k-slider"),
    html.Div(id="metrics"),
    dcc.Graph(id="pca-scatter")
    ]
)


In [ ]:
VimeoVideo("715724086", h="e9ed963958", width=600)


In [ ]:
VimeoVideo("715724816", h="80ec24d3d6", width=600)


In [ ]:
def get_high_var_features(trimmed=True, return_feat_names=True): 
    """Returns the five highest-variance features of ``df``.

    Parameters
    ----------
    trimmed : bool, default=True
        If ``True``, calculates trimmed variance, removing bottom and top 10%
        of observations.

    return_feat_names : bool, default=False
        If ``True``, returns feature names as a ``list``. If ``False``
        returns ``Series``, where index is feature names and values are
        variances.
    """
    if trimmed:
        top_five_features = (
        df.apply(trimmed_var).sort_values().tail(5)
    )
    
    else:
        top_five_features = df.var().sort_values().tail(5)

    # extract names
    if return_feat_names:
        top_five_features = top_five_features.index.to_list()

    
    return top_five_features


In [ ]:
VimeoVideo("715724735", h="5238a5c518", width=600)


In [ ]:
@app.callback(
    Output("bar-chart", "figure"),
    Input("trim-button", "value")
)

def serve_bar_chart(trimmed=True): 
    """Returns a horizontal bar chart of five highest-variance features.

    Parameters
    ----------
    trimmed : bool, default=True
        If ``True``, calculates trimmed variance, removing bottom and top 10%
        of observations.
    """
    # get features
    top_five_features = get_high_var_features(trimmed=trimmed, return_feat_names=False)

    # Build bar chart
    fig = px.bar(
        x=top_five_features,
        y=top_five_features.index, 
        orientation="h")
    
    fig.update_layout(
        xaxis_title="Variance",
        yaxis_title="Feature"
    )
    
    return fig


In [ ]:
serve_bar_chart(trimmed=True)


In [ ]:
VimeoVideo("715724706", h="b672dd9202", width=600)


In [ ]:
VimeoVideo("715724662", h="957a128506", width=600)


In [ ]:
VimeoVideo("715724573", h="7de7932f70", width=600)


In [ ]:
VimeoVideo("715725482", h="88aa75b1e2", width=600)


In [ ]:
VimeoVideo("715725430", h="5d24607b0c", width=600)


In [ ]:
VimeoVideo("715725405", h="8944b9c674", width=600)


In [ ]:
VimeoVideo("715725235", h="55229ebf88", width=600)


In [ ]:
def get_model_metrics(trimmed=True, k=2, return_metrics=False):  
    """Build ``KMeans`` model based on five highest-variance features in ``df``.

    Parameters
    ----------
    trimmed : bool, default=True
        If ``True``, calculates trimmed variance, removing bottom and top 10%
        of observations.

    k : int, default=2
        Number of clusters.

    return_metrics : bool, default=False
        If ``False`` returns ``KMeans`` model. If ``True`` returns ``dict``
        with inertia and silhouette score.

    """
    # Get high var features
    features = get_high_var_features(trimmed=trimmed, return_feat_names=True)

    # Create feature matrix
    X = df[features]

    # Build model
    model = make_pipeline(
        StandardScaler(),
        KMeans(n_clusters=k, random_state=42)
    )
    model.fit(X)
    
    if return_metrics:
        # Calculate inertia
        i = model.named_steps["kmeans"].inertia_

    # Calculate silhouette score
        ss = silhouette_score(X, model.named_steps["kmeans"].labels_)

    # Put results into dictionary
        metrics = {
            "inertia": round(i),
            "silhouette": round(ss, 3)
        }

    # Return dictionary to user
        return metrics
    
    return model


In [ ]:
VimeoVideo("715725137", h="124312b155", width=600)


In [ ]:
@app.callback(
    Output("metrics", "children"),
    Input("trim-button", "value"),
    Input("k-slider", "value")
    )

def serve_metrics(trimmed=True, k=2):  
    """Returns list of ``H3`` elements containing inertia and silhouette score
    for ``KMeans`` model.

    Parameters
    ----------
    trimmed : bool, default=True
        If ``True``, calculates trimmed variance, removing bottom and top 10%
        of observations.

    k : int, default=2
        Number of clusters.
    """
   # Get metrics
    metrics = get_model_metrics(trimmed=trimmed, k=k, return_metrics=True)

    # Add metrics to HTML elements
    text = [
        html.H3(f"Inertia: {metrics['inertia']}"),
        html.H3(f"Silhouette Score: {metrics['silhouette']}")
    ]

    return text


In [ ]:
VimeoVideo("715726075", h="ee0510063c", width=600)


In [ ]:
VimeoVideo("715726033", h="a658095771", width=600)


In [ ]:
VimeoVideo("715725930", h="f957d27741", width=600)


In [ ]:
def get_pca_labels(trimmed=True, k=2):  
    """
    ``KMeans`` labels.

    Parameters
    ----------
    trimmed : bool, default=True
        If ``True``, calculates trimmed variance, removing bottom and top 10%
        of observations.

    k : int, default=2
        Number of clusters.
    """
    # Create feature matrix
    features = get_high_var_features(trimmed=trimmed, return_feat_names=True)
    X = df[features]
    
    # Build transformer
    transformer = PCA(n_components=2, random_state=42)

    # Tranform data
    X_t = transformer.fit_transform(X)
    X_pca = pd.DataFrame(X_t, columns=["PC1", "PC2"])

    # Add labels
    model = get_model_metrics(trimmed=trimmed, k=k, return_metrics=False)
    X_pca["labels"] = model.named_steps["kmeans"].labels_.astype(str)
    X_pca.sort_values("labels", inplace=True)
    
    return X_pca


In [ ]:
VimeoVideo("715725877", h="21365c862f", width=600)


In [ ]:
@app.callback(
    Output("pca-scatter", "figure"),
    Input("trim-button", "value"),
    Input("k-slider", "value")
)

def serve_scatter_plot(trimmed=True, k=2):
    """Build 2D scatter plot of ``df`` with ``KMeans`` labels.

    Parameters
    ----------
    trimmed : bool, default=True
        If ``True``, calculates trimmed variance, removing bottom and top 10%
        of observations.

    k : int, default=2
        Number of clusters.
    """
    fig = px.scatter(
    data_frame=get_pca_labels(trimmed=trimmed, k=k),
        x="PC1",
        y="PC2",
        color="labels",
        title="PCA Representation of Clusters"
    )

    fig.update_layout(xaxis_title="PC1", yaxis_title="PC2")
    
    return fig


In [ ]:
VimeoVideo("715725777", h="4b3ecacb85", width=600)


In [ ]:
public_url = f"https://{os.environ['DW_CONTAINER_NAME']}-9000.{os.environ['DW_CONTAINER_HOST_NAME']}/"

app.run_server(host="0.0.0.0", port=9000, jupyter_server_url=public_url) 
